# 父子分块 + 元数据过滤：RAG 增强第一式（011）

> 配套章节：[04-分块与元数据](../../08-RAG体系/04-分块与元数据.md)、[05-查询侧技术](../../08-RAG体系/05-查询侧技术.md)；语料复用 [009-中文RAG全管线](009-中文RAG全管线.ipynb)。
> 一句话：**索引粒度等于答案粒度——用「子块」建索引（找得准），用「父块」回填上下文（喂得足），再让元数据把检索关进专属主题。**

两个问题按一个难点：

1. **粒度谜题**：文档太大→embedding 被「平均化」，具体问题找准？文档太小→上下文差一截，跨子句线索全丢。父子分块 = **检索用小块、回填用大块**，鱼与熊掌通吃。
2. **噪音污染**：top-5 里混着旁类文档。给每个块挂主题元数据，过滤后检索直接被「关进」用户所属的主题。

> 实现说明（诚实标注）：为不下载长文档，本 notebook 把前 90 条语料每 3 条拼成 1 篇「章」（30 篇 × 3 子块）——拼接是合成的，但「粒度越细检索越准」的机制与数字是真实的，可一键复现。


## 1. 安装依赖（Colab 已含大部分）

In [1]:
import importlib, subprocess, sys
for pkg in ("rank_bm25", "jieba"):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
try:
    import faiss
except ImportError:
    for pkg in ("faiss", "faiss-cpu"):
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
            import faiss
            break
        except Exception:
            continue
print("deps ok")

deps ok


## 2. 数据：复用 009 的 100 条中文百科语料

第 i 条 gold 问题答案在第 i 条上下文里；本 notebook 只用前 90 条做两级结构。

In [2]:
# 100 条中文百科上下文 + 每条一个 gold 问题（NOTEBOOK 内嵌，可直接跑）
CORPUS = [
 [
  "引力波是时空弯曲的涟漪，由大质量天体加速运动时产生，2015年首次被LIGO探测到",
  "引力波是2015年被哪个实验首次探测到的"
 ],
 [
  "量子纠缠指两个粒子无论相隔多远，其量子态都会相互关联，是量子通信的基础",
  "量子纠缠的特性和在通信中的作用是什么"
 ],
 [
  "Transformer是2017年提出的基于注意力机制的网络结构，是大语言模型的基础",
  "Transformer的核心机制是什么"
 ],
 [
  "梯度下降通过沿损失函数负梯度方向迭代更新参数，是训练神经网络的最基本优化算法",
  "训练神经网络最基本的优化算法是什么"
 ],
 [
  "图灵测试由阿兰·图灵在1950年提出，用于判断机器是否具备人类智能",
  "图灵测试是在哪一年由谁提出的"
 ],
 [
  "卷积神经网络通过卷积核在局部区域提取特征，特别适合图像识别任务",
  "卷积神经网络在什么任务上表现出色"
 ],
 [
  "反向传播算法通过链式法则逐层计算梯度，是深度学习训练的核心机制",
  "链式法则在深度学习训练中如何被应用"
 ],
 [
  "元学习的目标是让模型学会学习，在少量样本上快速适应新任务",
  "让模型从少量样本快速适应新任务的方法是什么"
 ],
 [
  "注意力机制允许模型在做预测时动态聚焦输入序列中最相关的部分",
  "注意力机制的核心思想是什么"
 ],
 [
  "大规模预训练语言模型通过在海量文本上预测下一个词来学习语言规律",
  "语言模型通过什么任务来学习语言规律"
 ],
 [
  "强化学习通过奖励信号引导智能体在环境中探索并优化策略",
  "强化学习中奖励信号的作用是什么"
 ],
 [
  "决策树通过递归划分特征空间来对数据进行分类或回归",
  "决策树的工作原理是什么"
 ],
 [
  "支持向量机通过寻找最大间隔超平面来分割不同类别的样本",
  "支持向量机的核心思路是什么"
 ],
 [
  "贝叶斯定理描述了在已知先验概率和观测数据后更新信念的方法",
  "贝叶斯定理主要用于做什么"
 ],
 [
  "过拟合是指模型在训练集上表现好但在新数据上表现差的现象",
  "什么是过拟合"
 ],
 [
  "K均值聚类通过迭代把样本划分为K个簇，使簇内距离最小",
  "K均值聚类的目标是什么"
 ],
 [
  "主成分分析通过线性变换把高维数据投影到低维，保留最大方差",
  "主成分分析的作用是什么"
 ],
 [
  "正则化通过在损失函数中加入模型复杂度惩罚项来抑制过拟合",
  "正则化的作用是什么"
 ],
 [
  "学习率决定了参数更新的步长，过大导致震荡过小导致收敛慢",
  "学习率过大会导致什么问题"
 ],
 [
  "批归一化通过在每层对激活值做归一化，加速深度学习训练收敛",
  "批归一化在训练中起什么作用"
 ],
 [
  "残差网络通过跳连接缓解深层网络训练中的梯度消失问题",
  "残差网络如何解决深层训练中的问题"
 ],
 [
  "长短期记忆网络通过门控机制解决传统RNN的长期依赖问题",
  "LSTM解决的核心问题是什么"
 ],
 [
  "知识蒸馏通过让学生模型模仿教师模型的软输出，把大模型能力压缩到小模型",
  "知识蒸馏的目标是什么"
 ],
 [
  "剪枝通过移除神经网络中不重要的权重或神经元来压缩模型",
  "神经网络剪枝的做法是什么"
 ],
 [
  "量化通过把模型权重从浮点数转换到更低比特表示来减小模型体积",
  "模型量化的目的是什么"
 ],
 [
  "联邦学习让多个设备在本地训练模型，只上传梯度而不共享原始数据",
  "联邦学习的核心特点是只上传什么"
 ],
 [
  "对比学习通过拉近正样本对、推远负样本对来学习良好表示",
  "对比学习的训练目标是什么"
 ],
 [
  "自编码器通过无监督方式学习数据的压缩表示并重构输入",
  "自编码器的用途是什么"
 ],
 [
  "生成对抗网络由生成器和判别器博弈训练，可生成逼真图像",
  "生成对抗网络包含哪两个部分"
 ],
 [
  "扩散模型通过逐步去噪从随机噪声生成数据，是图像生成的主流方法",
  "扩散模型的生成过程是怎样的"
 ],
 [
  "词向量用低维稠密向量表示词义，可通过余弦相似度衡量词间语义相近程度",
  "如何衡量两个词的语义相近程度"
 ],
 [
  "N-gram语言模型基于马尔可夫假设，假设当前词只依赖前n-1个词",
  "N-gram语言模型的马尔可夫假设是什么"
 ],
 [
  "BPE分词算法通过合并最频繁的字符对来构建词表，能有效处理未登录词",
  "BPE分词算法的核心操作是什么"
 ],
 [
  "词嵌入通过训练把词映射到向量空间，使语义相近的词距离更近",
  "词嵌入的目标是什么"
 ],
 [
  "提示工程通过精心设计指令引导大模型输出理想结果，是应用大模型的核心技能",
  "提示工程的主要意义是什么"
 ],
 [
  "思维链提示让模型分步骤推理问题，能显著提高复杂推理任务的表现",
  "思维链提示如何提高模型的推理能力"
 ],
 [
  "检索增强生成通过外部知识库检索相关文档来增强模型回答的准确性",
  "检索增强生成解决什么问题"
 ],
 [
  "向量数据库存储并检索高维向量，支持近似最近邻搜索，是RAG应用的基础设施",
  "向量数据库支持哪种搜索"
 ],
 [
  "限流算法通过令牌桶或滑动窗口控制系统在单位时间内处理的请求数",
  "常用的限流算法有哪些"
 ],
 [
  "数据库索引以B+树或哈希等结构加速数据查询，是数据库性能优化的核心",
  "数据库索引的主要作用是什么"
 ],
 [
  "缓存通过存储热点数据避免重复计算，从而大幅降低系统延迟",
  "缓存为何能降低系统延迟"
 ],
 [
  "负载均衡把请求分发到多个服务器，防止单点过载并提升可用性",
  "负载均衡的作用是什么"
 ],
 [
  "消息队列通过异步解耦组件，在流量高峰时削峰填谷，保证系统稳定",
  "消息队列在流量高峰时如何起作用"
 ],
 [
  "容器通过操作系统级虚拟化隔离应用，使部署快速且环境一致",
  "容器的核心优势是什么"
 ],
 [
  "分布式系统通过多台机器协同计算，在单个节点故障时仍能继续工作",
  "分布式系统的目标之一是容错具体是指什么"
 ],
 [
  "数据库事务保证一组操作要么全部成功要么全部回滚，具备原子性",
  "数据库事务的原子性是什么意思"
 ],
 [
  "布隆过滤器用位数组表示集合成员，占用极少的空间但允许小概率误判",
  "布隆过滤器在空间和准确性上有什么特点"
 ],
 [
  "一致性哈希在节点加入或退出时只影响少量数据迁移，是分布式缓存常用方案",
  "一致性哈希的优点是什么"
 ],
 [
  "断点续传通过记录已上传分片位置，在网络中断后能从中断处继续传输",
  "断点续传的原理是什么"
 ],
 [
  "压缩算法通过消除数据冗余降低存储和传输成本，是系统优化常用手段",
  "压缩算法降低的成本有哪些"
 ],
 [
  "视线跟踪结合眼动和头部姿态估计注视点，被用于用户交互和注意力分析",
  "人机交互中视线跟踪的应用是什么"
 ],
 [
  "中医学以阴阳五行理论为基础，通过辨证论治指导临床实践",
  "中医辨证论治在临床实践中的作用是什么"
 ],
 [
  "量子计算利用量子叠加和纠缠特性，在特定问题上可远超经典计算机",
  "量子计算为什么能超越经典计算机"
 ],
 [
  "黑洞是引力极强连光也无法逃逸的天体，其边界称为事件视界",
  "黑洞连什么也无法逃逸"
 ],
 [
  "光合作用是植物利用光能合成有机物的过程，是生态系统能量流动的起点",
  "光合作用在生态系统中扮演什么角色"
 ],
 [
  "人类基因组计划旨在确定人类基因组的全部DNA序列，是生命科学的基础工程",
  "人类基因组计划的目标是什么"
 ],
 [
  "疫苗通过刺激免疫系统产生记忆性应答，可在病原入侵时快速防御",
  "疫苗预防疾病的原理是什么"
 ],
 [
  "全球变暖主要由温室气体排放导致，会引发海平面上升和极端天气增多",
  "全球变暖的主要成因和影响有哪些"
 ],
 [
  "区块链通过哈希链和共识机制保证数据的不可篡改性，比特币是其著名的应用",
  "区块链保证数据不可篡改机制的叫什么"
 ],
 [
  "5G网络具备低时延高速率大连接的特性，是工业互联网和自动驾驶的关键",
  "5G网络的重要特性有哪些"
 ],
 [
  "自动驾驶通过传感器融合和决策规划，在复杂交通环境中安全驾行",
  "自动驾驶系统依赖哪两类核心技术"
 ],
 [
  "脑机接口通过采集脑电信号并经解码控制外部设备，可用于瘫痪患者康复",
  "脑机接口的应用场景有哪些"
 ],
 [
  "基因编辑技术CRISPR能精准修改特定基因位点，在疾病治疗上潜力巨大",
  "CRISPR技术的核心能力是什么"
 ],
 [
  "人造太阳托卡马克装置通过磁场约束等离子体，探索可控核聚变能源",
  "托卡马克装置的目标是什么"
 ],
 [
  "超级计算机用加快气象预报药物研发等科学计算，其算力通常用浮点运算次数衡量",
  "超级计算机的算力通常用什么衡量"
 ],
 [
  "湿地被称为地球之肾，在净化水质调蓄洪水维护生物多样性方面发挥关键作用",
  "湿地为什么被称为地球之肾"
 ],
 [
  "青藏高原被称为亚洲水塔，是长江黄河等大河的源头",
  "亚洲水塔指的是哪里"
 ],
 [
  "大熊猫的食性已特化为以竹子为主，消化系统仍保留肉食动物的特征",
  "大熊猫的食性特点是什么"
 ],
 [
  "福建土楼多为客家人所建，以厚墙圆形或方形布局在防御和宗族聚居上独具特色",
  "福建土楼的建造者和特点是什么"
 ],
 [
  "都江堰是李冰父子主持修建的水利工程，引水灌溉成都平原两千多年",
  "都江堰水利工程是谁主持修建的"
 ],
 [
  "清明上河图描绘北宋都城汴京的市井生活，是研究宋代社会的重要史料",
  "清明上河图描绘的是哪个朝代的都城"
 ],
 [
  "彗星主要由冰和尘埃组成，接近太阳时会形成长长的彗尾",
  "彗星主要由什么组成"
 ],
 [
  "极光是太阳风带电粒子撞击高层大气分子产生的发光现象",
  "极光是如何产生的"
 ],
 [
  "海啸通常由海底地震或火山喷发引发，目前只能预警难以完全防御",
  "海啸通常由什么引发"
 ],
 [
  "沙漠化是土地因过度放牧开垦等原因退化，严重威胁粮食安全",
  "沙漠化的主要原因有哪些"
 ],
 [
  "碳中和指通过减排和碳汇使二氧化碳排放量达到平衡，是全球气候目标",
  "碳中和的目标是什么"
 ],
 [
  "芯片制程越小晶体管密度越高功耗越低，是半导体产业竞争的核心指标",
  "芯片制程缩小的意义是什么"
 ],
 [
  "熔断机制在股票指数跌至阈值时暂停交易，防止市场恐慌式下跌",
  "股市熔断机制的设置目的是什么"
 ],
 [
  "量化交易通过数学模型和程序化下单捕捉市场套利与回撤机会",
  "量化交易的特点是什么"
 ],
 [
  "复利俗称利滚利，指利息自产生起再计入本金继续生息",
  "复利的通俗说法是什么"
 ],
 [
  "市盈率是股价与每股收益的比值，估值高低需结合行业成长性来看",
  "市盈率是哪个指标与每股收益的比值"
 ],
 [
  "分散投资把资金配置到不同类型的资产以降低单一资产下跌的冲击",
  "投资组合分散投资的目的是什么"
 ],
 [
  "通货膨胀指货币购买力下降物价总水平持续上升，美联储通过加息收水控制通胀",
  "应对通胀通常采用的货币政策工具是什么"
 ],
 [
  "碳交易市场给二氧化碳排放定价，让减产排放的企业能出售配额获利",
  "碳交易市场起什么作用"
 ],
 [
  "供应链安全指关键原材料和零部件的供应稳定，是制造业的核心关切",
  "供应链安全主要指什么"
 ],
 [
  "反应堆堆芯需持续冷却，一旦失去冷却将可能导致堆芯熔毁事故",
  "核电站反应堆失去冷却可能导致的后果是什么"
 ],
 [
  "空间站需要氧气水等生命保障系统为航天员长时间驻留创造条件",
  "空间站的生命保障系统为航天员提供什么"
 ],
 [
  "探索合成生物学通过改造基因回路让微生物生产药物燃料等物质",
  "合成生物学的应用方向有哪些"
 ],
 [
  "柔性电子使设备可弯曲折叠，是未来可穿戴设备的重要方向",
  "柔性电子的主要优势是什么"
 ],
 [
  "星链通过低轨卫星组网为地面提供高速网络，特点是延迟低覆盖广",
  "星链的网络特征是什么"
 ],
 [
  "长征火箭是中国进入太空的主要运载工具，探索月球和空间站建设依赖它",
  "长征火箭的作用是什么"
 ],
 [
  "前庭觉负责感知头部的倾斜和旋转，是平衡系统的关键",
  "前庭觉在人体中起什么作用"
 ],
 [
  "激素由内分泌腺分泌，随血液循环到达靶器官调节生理活动",
  "激素是如何运输到靶器官的"
 ],
 [
  "免疫系统的记忆细胞能在二次感染时快速产生更强的免疫应答",
  "免疫记忆细胞在二次感染时有何表现"
 ],
 [
  "人工智能对齐指让模型的价值观与人类意图一致，是安全部署大模型的关键",
  "什么是人工智能对齐"
 ],
 [
  "幻觉是生成式模型输出与事实不符内容的现象，需要通过检索验证等手段缓解",
  "生成式模型输出与事实不符的现象叫什么"
 ],
 [
  "提示注入是通过恶意指令让大模型违反设定，是LLM应用的主要安全威胁",
  "提示注入攻击的原理是什么"
 ],
 [
  "联邦蒸馏结合联邦学习与知识蒸馏，在保护隐私的同时聚合各端模型能力",
  "联邦蒸馏结合了哪两种技术"
 ],
 [
  "边缘计算把计算任务下沉到靠近数据源的设备，降低时延减少带宽消耗",
  "边缘计算降低的是什么"
 ],
 [
  "RDMA允许数据绕过操作系统内核直达网卡内存，显著降低网络延迟",
  "RDMA降低网络延迟的原理是什么"
 ]
]
corpus_q = [c for c, _ in CORPUS]     # 检索语料 = 全部上下文
test_q = [q for _, q in CORPUS]       # 测试问题 = 全部 gold 问题
gold_idx = list(range(len(corpus_q))) # 第 i 条问题的答案就在第 i 个上下文里
print("语料条目:", len(corpus_q), " 测试问题:", len(test_q))

语料条目: 100  测试问题: 100


## 3. 组成「章 → 子块」两级结构

每 3 条连成 1 篇「章」，内部每个条目就是一个子块：`章[i] = 子块[3i,3i+1,3i+2]`。

In [3]:
CH = 3                 # 每章子块数
NSECT = 90             # 用前 90 条（30 章）
sects = CORPUS[:NSECT]
par = [" ".join(c[0] for c in sects[k:k + CH]) for k in range(0, NSECT, CH)]   # 30 篇「章」
ques = [q for _, q in sects]                                                    # 90 个子块问题
print("章数:", len(par), " 子块数:", len(sects))
print("示例 章[0] =", par[0][:36] + " … ｜ 子块[0] =", sects[0][0][:24] + " …")

章数: 30  子块数: 90
示例 章[0] = 引力波是时空弯曲的涟漪，由大质量天体加速运动时产生，2015年首次被LI … ｜ 子块[0] = 引力波是时空弯曲的涟漪，由大质量天体加速运动时产 …


## 4. 两级索引：子块索引 + 章（父块）索引

同一套 bge 编码，两个 faiss 内积索引——父子分块的实现就这么点事。

In [4]:
import numpy as np, faiss
from sentence_transformers import SentenceTransformer
import jieba

model = SentenceTransformer("BAAI/bge-small-zh-v1.5")
def emb(txts):
    return model.encode(txts, normalize_embeddings=True, show_progress_bar=False)

child_emb = emb([c[0] for c in sects])          # 子块 90 维→90 条
parent_emb = emb(par)                            # 章  30 条
ci = faiss.IndexFlatIP(child_emb.shape[1]); ci.add(np.ascontiguousarray(child_emb))
pi = faiss.IndexFlatIP(parent_emb.shape[1]); pi.add(np.ascontiguousarray(parent_emb))

def topn(idx, n, q):
    v = model.encode([q], normalize_embeddings=True)
    _, k = idx.search(np.ascontiguousarray(v), n)
    return k[0].tolist()
print("子块 faiss:", ci.ntotal, "条 ｜ 父块 faiss:", pi.ntotal, "条")

子块 faiss: 90 条 ｜ 父块 faiss: 30 条


## 5. 实验①：粒度——「找到哪一章」≠「翻到哪一句」

24 个随机子块问题（种子固定可复现）。**整篇索引**报「这篇对」/「这篇错」，**子块索引**报「这句对/句在 top-3」。

关键指标：**证据浓度**——整篇返回 3 个块但只有 1 块相关（1/3）；子块返回 1 块就是证据（1/1）。


In [5]:
rng = np.random.RandomState(0)
tq = rng.choice(np.arange(NSECT), 24, replace=False).tolist()

par_hit = [int(topn(pi, 1, ques[i])[0] == i // CH) for i in tq]
ch1 = [int(topn(ci, 1, ques[i])[0] == i) for i in tq]
ch3 = [int(i in topn(ci, 3, ques[i])) for i in tq]
print("=== 实验① 粒度（整篇 30 篇 vs 子块 90 块，24 问） ===")
print("整篇检索 top-1 命中「所在章」 : %d/24  %.2f   ← 拼接章被「平均化」，具体问题选错章一半以上" % (sum(par_hit), sum(par_hit) / 24))
print("子块检索 top-1 命中「gold 子块」: %d/24  %.2f" % (sum(ch1), sum(ch1) / 24))
print("子块检索 gold∈top-3           : %d/24  %.2f" % (sum(ch3), sum(ch3) / 24))
print("上下文成本：整篇=3×子块 token；证据浓度 整篇 1/3 vs 子块 1/1")
print()
print("注：本语料拼接「章」非自然长文，但与真实长文档同理——块越大，单条 embedding")
print("    越背不动具体问题（具体问题检索漂移）。产线上见到的「doc 太大检索就飘」同一现象。")
print()
# 展示一条整篇选错章、子块扳正的例子
i = [x for x in tq if x // CH != topn(pi, 1, ques[x])[0]][0]
print("示例（整篇选错章 → 子块扳正）：")
print("  问题      :", ques[i])
print("  所在章[%d] :" % (i // CH), par[i // CH][:30] + "…")
print("  整篇 top1 → 章[%d] %s" % (topn(pi, 1, ques[i])[0], par[topn(pi, 1, ques[i])[0]][:30] + "…"))
print("  子块 top1 → 子块[%d] %s" % (topn(ci, 1, ques[i])[0], sects[topn(ci, 1, ques[i])[0]][0][:30] + "…"))

=== 实验① 粒度（整篇 30 篇 vs 子块 90 块，24 问） ===
整篇检索 top-1 命中「所在章」 : 16/24  0.67   ← 拼接章被「平均化」，具体问题选错章一半以上
子块检索 top-1 命中「gold 子块」: 24/24  1.00
子块检索 gold∈top-3           : 24/24  1.00
上下文成本：整篇=3×子块 token；证据浓度 整篇 1/3 vs 子块 1/1

注：本语料拼接「章」非自然长文，但与真实长文档同理——块越大，单条 embedding
    越背不动具体问题（具体问题检索漂移）。产线上见到的「doc 太大检索就飘」同一现象。

示例（整篇选错章 → 子块扳正）：
  问题      : Transformer的核心机制是什么
  所在章[0] : 引力波是时空弯曲的涟漪，由大质量天体加速运动时产生，2015…
  整篇 top1 → 章[2] 反向传播算法通过链式法则逐层计算梯度，是深度学习训练的核心机…
  子块 top1 → 子块[2] Transformer是2017年提出的基于注意力机制的网络…


## 6. 父子回填：检索小、回填大

子块命中后，**回填它所在的章**作为上下文——找得准（子块索引）还喂得足（同章 3 块都进来）。对 24 问实测两件事：回填把上下文放大了多少、gold 子块是否仍在回填上下文内。

In [6]:
covered = 0
for i in tq:
    j = topn(ci, 1, ques[i])[0]
    parent_i = j // CH
    covered += int(i in range(parent_i * CH, parent_i * CH + CH))
ratio = [len(par[i // CH]) / len(sects[i][0]) for i in tq]   # 回填父块 / 单子块
print("=== 实验①b 父子回填（24 问） ===")
print("子块 top-1 命中 gold 子块：%d/24（本语料顶格——子块索引选句很准）" % sum(ch1))
print("回填父块后 gold 仍在上下文中：%d/24" % covered)
print("上下文放大：回填父块/子块 平均 %.2f×（同章 3 块全给，跨子句线索不会丢）" % (
    sum(ratio) / len(ratio)))
print()
print("跨子句的价值：若答案要跨两个子块才能拼出来（真实长文档很常见），只给子块")
print("  上下文会缺一半证据；回填父块把同章邻居一并带上。")
print("  生产组合：父子回填 + Reranker/LLM 选择（见 09-Advanced-RAG 章节）。")

=== 实验①b 父子回填（24 问） ===
子块 top-1 命中 gold 子块：24/24（本语料顶格——子块索引选句很准）
回填父块后 gold 仍在上下文中：24/24
上下文放大：回填父块/子块 平均 3.14×（同章 3 块全给，跨子句线索不会丢）

跨子句的价值：若答案要跨两个子块才能拼出来（真实长文档很常见），只给子块
  上下文会缺一半证据；回填父块把同章邻居一并带上。
  生产组合：父子回填 + Reranker/LLM 选择（见 09-Advanced-RAG 章节）。


## 7. 实验②：主题（元数据）过滤

给每个子块挂主题标签（5 类主题词表，子块文本命中最多的类即其主题；未命中 = 其他）。然后取 6 条「问题里自带主题词」的题，对比 no-filter 与 filter 的 top-5 **主题纯度**。

In [7]:
from collections import Counter
TOPICS = [
    ("AI/机器学习", ["模型", "算法", "学习", "人工智能"]),
    ("通信/网络",   ["网络", "延迟", "带宽", "通信"]),
    ("生物/医学",   ["细胞", "免疫", "基因", "疾病", "RNA", "细菌", "疗法"]),
    ("太空/天文",   ["太空", "卫星", "火箭", "宇宙", "天体", "登月", "空间站"]),
    ("金融/经济",   ["股票", "利率", "通胀", "货币", "资产", "投资"]),
]
def topic_of(text):
    hits = Counter()
    for name, kws in TOPICS:
        n = sum(text.count(w) for w in kws)
        if n:
            hits[name] = n
    return hits.most_common(1)[0][0] if hits else None

labels = [topic_of(c[0]) for c in sects]
print("打标分布:", Counter(x for x in labels if x))

def filter_search(q, keep, n=5):
    ids = topn(ci, NSECT, q)
    out = []
    for x in ids:
        if labels[x] == keep:
            out.append(x)
        if len(out) == n:
            break
    return out[:n]

sel = []
for i in range(NSECT):
    L = labels[i]
    if L and any(kw in ques[i] for kw in dict(TOPICS)[L]):
        sel.append((i, L))
rng2 = np.random.RandomState(1)
chosen = rng2.choice(len(sel), 6, replace=False).tolist()
print()
print("=== 实验② 主题过滤（6 题，主题脑里有明确归属） ===")
stats = []
for ix in chosen:
    i, L = sel[ix]
    n5 = topn(ci, 5, ques[i])
    nofil = [labels[x] for x in n5]
    filt = filter_search(ques[i], L)
    pure_no = sum(1 for x in nofil if x == L)
    gold_hit = int(i in filt)
    stats.append((pure_no, len(filt), gold_hit))
    print("题[%2d/%s] no-filter top5 主题=%s → filter 后 %d 条全属 %s, gold∈top? %d" % (
        i, L, nofil, len(filt), L, gold_hit))
print()
print("no-filter top-5 平均主题纯度: %.2f（5 个里只有 %.1f 个是目标主题）"
      % (sum(s[0] for s in stats) / len(stats) / 5, sum(s[0] for s in stats) / len(stats)))
print("filter 后 top-5 主题纯度: 1.00（同名限定）；gold 命中 %d/6（过滤保精度、不涨召回）"
      % sum(s[2] for s in stats))

# 落盘汇总
import pathlib, pandas as pd
pathlib.Path("out").mkdir(exist_ok=True)
pd.DataFrame([{"query": ques[sel[ix][0]], "topic": sel[ix][1],
               "no_filter_purity": stats[k][0] / 5, "filter_hit": stats[k][2]}
              for k, ix in enumerate(chosen)]).to_csv(
    "out/011_parent_child_metadata.csv", index=False, encoding="utf-8-sig")
print("\n已保存 out/011_parent_child_metadata.csv")

打标分布: Counter({'AI/机器学习': 26, '通信/网络': 9, '生物/医学': 4, '太空/天文': 3, '金融/经济': 3})

=== 实验② 主题过滤（6 题，主题脑里有明确归属） ===
题[38/AI/机器学习] no-filter top5 主题=['AI/机器学习', None, None, 'AI/机器学习', None] → filter 后 5 条全属 AI/机器学习, gold∈top? 1
题[31/AI/机器学习] no-filter top5 主题=['AI/机器学习', 'AI/机器学习', 'AI/机器学习', None, 'AI/机器学习'] → filter 后 5 条全属 AI/机器学习, gold∈top? 1
题[40/通信/网络] no-filter top5 主题=['通信/网络', 'AI/机器学习', None, None, None] → filter 后 5 条全属 通信/网络, gold∈top? 1
题[ 6/AI/机器学习] no-filter top5 主题=['AI/机器学习', 'AI/机器学习', 'AI/机器学习', 'AI/机器学习', 'AI/机器学习'] → filter 后 5 条全属 AI/机器学习, gold∈top? 1
题[81/金融/经济] no-filter top5 主题=['金融/经济', None, None, 'AI/机器学习', None] → filter 后 3 条全属 金融/经济, gold∈top? 1
题[25/AI/机器学习] no-filter top5 主题=['AI/机器学习', 'AI/机器学习', 'AI/机器学习', 'AI/机器学习', '通信/网络'] → filter 后 5 条全属 AI/机器学习, gold∈top? 1

no-filter top-5 平均主题纯度: 0.57（5 个里只有 2.8 个是目标主题）
filter 后 top-5 主题纯度: 1.00（同名限定）；gold 命中 6/6（过滤保精度、不涨召回）

已保存 out/011_parent_child_metadata.csv
